# Chapter 2: Explore STAC Catalog and Read NetCDF files using STAC
* **Creators:** Ann Windnagel and Robyn Marowitz
* **Affiliation:** [National Snow and Ice Data Center (NSIDC)](https://nsidc.org/home "National Snow and Ice Data Center")
* **History:**
  * October 2026 - Updating to work with V6 SIC CDR
  * January 2025 - Initial version

## Overview

In this notebook (Notebook 2), you will learn what a SpatioTemporal Asset Catalog (STAC) is. You will also learn how to explore the features and variables of the SIC CDR STAC Collection and Items and open and read the SIC CDR netCDF files using PySTAC.

### Prerequisites
Read and understand the material in Notebook 1.

### Targeted Level
Beginner

### Learning Outcomes
Once you have completed this notebook, you will be able to:
* Understand what a STAC catalog is
* Open the SIC CDR STAC collection and extract the content list using `PySTAC`
* Explore the *Collection* and its *Items*, properties, and metadata
* Open and read a SIC CDR netCDF file from the STAC using `Xarray`

### Time Estimate
TBD

## What is a STAC Catalog?

A [SpatioTemporal Asset Catalog (STAC)](https://stacspec.org/en "Spatio Temporal Asset Catalog Web Site") is a common language specification used to describe geospatial data. It improves the accessibility and discoverability of the data it describes. A STAC catalog provides special metadata about the collection that provides a standardized way to expose the spatio-temporal data and allows it to be more easily worked with, indexed, and discovered. A STAC catalog uses a JSON (JavaScript Object Notation) schema to describe the data via different STAC specifications.

<div class="alert alert-block alert-success">
<h3><strong>Explore</strong></h3>
<h4>Check out other STAC resources</h4>
If you would like to read more about STAC or see other examples visit:
 <ul>
     <li><a href="https://stacspec.org/en">STAC Specification</a>: This is the official STAC specifcatios web site. It provides information for data providers, developers, and data users. Here, you can find more information on what STAC is and how to create you own, including tutorials on how to get started with STAC.</li>
     <li><a href="https://stacindex.org/">STAC Index</a>: This web site is an extension of stacspec.org. Here you can find other examples of STAC catalogs from all science disciplines as well as software and tools to work with STAC and other learning resources.</li>
     <li><a href="https://github.com/radiantearth/stac-spec">STAC on GitHub</a>: This repository contains the core object type specifications, examples, validation schemas, and documentation about the STAC specification.</li>
 </ul>
</div>


A STAC (SpatioTemporal Asset Catalog) catalog describes data using a standard set of metadata specifications. For the SIC CDR STAC, NSIDC used the *Collection* and *Item* specifications. The *Collection* specification describes a group of related Items using a set of common metadata fields. In this case, the *Collection* is the entire SIC CDR data product. The *Item* specification describes each individual file in the *Collection*, which for the SIC CDR is each netCDF file. Together, these two specifications provide product-level (*Collection*) and file-level (*Item*) metadata for the data set.

The SIC CDR STAC catalog starts with a single JSON file, `collection.json`, that defines the *Collection*. Rather than storing each *Item* as its own JSON file, the *Items* are stored in two GeoParquet files, one for daily data and one for monthly data. Each row in these files is one *Item*, representing a single netCDF file for one hemisphere and one day or month. Both hemispheres are in the same file, so you can filter by the <mark>`noaa_cdr:hemisphere` property</mark>. The GeoParquet files are linked from the *Collection* as the `daily-items-geoparquet` and `monthly-items-geoparquet` assets. This format lets you load and filter the full set of *Items* at once with tools like **<mark>`PyArrow`, `geopandas`**</mark> and `stac-geoparquet`, instead of requesting thousands of individual JSON files. Each *Item* links to its netCDF file, which you can open directly over HTTPS without downloading it.

NSIDC used a common template as a starting point for the SIC CDR STAC from <a href="https://github.com/stactools-packages/stactools-packages.github.io">stactools-packages</a> on GitHub.

<div class="alert alert-block alert-success">
<h3><strong>Explore</strong></h3>
<h4>Check out the STAC JSON files</h4>
If you would like to view the sea ice CDR STAC JSON files before you work with them programmatically, you can see them here: <a href="https://noaadata.apps.nsidc.org/NOAA/G02202_V6/stac/" title="Sea Ice Concentration Climate Data Record STAC Catalog">https://noaadata.apps.nsidc.org/NOAA/G02202_V6/stac/</a>.
    
You can also explore the code NSIDC used to create the CDR STAC on GitHub: <a href="https://github.com/nsidc/noaa-cdr" title="NSIDC CDR STAC GitHub Repo">https://github.com/nsidc/noaa-cdr</a>.
</div>

### Import Relevant Packages
You will use [pystac](https://pystac.readthedocs.io/en/stable/ "Pystac documentation") to explore the sea ice CDR STAC *Collection* and *Items* and to view their details. [Xarray](https://docs.xarray.dev/en/stable/ "Xarray documentation") makes working with labelled multi-dimensional arrays, such as netCDF, in Python simple and efficient. [Fsspec](https://filesystem-spec.readthedocs.io/en/latest/ "Fsspec documentation") is a pythonic file system interface to make it easier to interact with file systems. <mark>Check through that these are all needed</mark>

#### Cell 1

In [1]:
# Cell 1
#import io
import fsspec
from fsspec.implementations.http import HTTPFileSystem
import pystac
#import pyarrow.parquet as pq
#from pyarrow import fs
import stac_geoparquet
#import requests
import geopandas as gpd
import xarray as xr

************************
## Tutorial Part 1

Read and explore STAC Collection
*************************

### Read the STAC Collection File

To begin, you will read the sea ice CDR STAC `collection.json` file into a `pystac collection object`, which you will assign to a variable called `collection`. Then, you can explore it using `pystac`.

#### Cell 2

In [2]:
# Cell 2
# Specify the URL to the sea ice CDR collection.json file on NSIDC servers
collection_url = "https://noaadata.apps.nsidc.org/NOAA/G02202_V6/stac/sea-ice-concentration/collection.json"

# Read file with pystact to create a pytsac collection object and assign to a variable
collection = pystac.Collection.from_file(collection_url)

### Explore Collection Information

Now that you've read in the *Collection*, in the following cells, you will look at some basic information and metadata about the *Collection*. There are different ways to extract this metadata information. You can view all the metadata at once by simply calling an instance of the collection object or you can extract specifc details you are interested in by calling attributes of the collection object. 

<mark>OLD V4 text</mark>

<mark>#### List All *Items* in the *Collection*
Using the `describe` method on the `collection` variable created above, you can display all of the *Items* for this *Collection*. The sea ice CDR data spans 1978 through the present year. You will see an item listed for each year and hemisphere for the daily files, plus one monthly file for each hemisphere that contains the whole time series.</mark>

<mark>In the `Item id`, there is a wealth of information due to its consistent format:
`seaice_conc_TIME_HH_YYYY[MM]_VXXRXX` 
where TIME is `daily` or `monthly`, HH is the hemisphere and can be `nh` for Northern Hemisphere or `sh` for Southern Hemisphere, YYYY is the 4-digit year of the data, MM is the 2-digit month (used for the monthly files only), and VXXRXX is the version (you will use version v04r00 in these notebooks).
</mark>

#### Print Out *Collection* Metadata

View all the metadata by simply calling the instance of the `pystac collection object`.

<div class="alert alert-block alert-success">
<h3><strong>Explore</strong></h3>
After you run the command in the next cell, you will see output about the <em>Collection</em>. Lines with a triangle next to them are clickable and reveal more metadata elements underneath. Click these and explore the contents of the <em>Collection</em>.
</div>

<div class="alert alert-block alert-info">
<strong>Note:</strong> If you open the <code>assests</code> attribute (not <code>item_assets</code>), you will see that the Collections Assets consists of only two GeoParquet files (one for the daily data and one for the monthly data) as described in the <em>What is a STAC Catalog?</em> section above.
</div>

#### Cell 3

In [3]:
# Cell 3
# Call instance of the pystac collection object to see all attributes
collection

<Collection id=noaa-cdr-sea-ice-concentration>

#### Print Specific *Collection* Attributes

In the Cell 3 above, you saw how to view all the metadata attributes of the *Collection* at once. You can also view specific attributes individually. In the Cell 4 below, print out the *Colleciton* `id`, `title`, `description`, and `assets` attributes. Also print the STAC version, which requires the use of the `get_stac_version` method. Finally, print some of the attributes for the assets themselves.

<div class="alert alert-block alert-info">
<strong>Note:</strong> You can extract information about the Collection assets such as the URL. This will be important later when you load the assets to extract data.
</div>

#### Cell 4

In [4]:
# Cell 4
# Extract only specific attributes from the Collection
# Print Collection title, description, temporal coverage, and assests
print('Collection ID : ',collection.id, "\n")
print("Collection Title:", collection.title, "\n")
print("Collection Description:", collection.description, "\n")
# Print STAC Version
print('STAC Version: ', pystac.get_stac_version(), "\n")
# print("Temporal Coverage:", collection.extent.temporal.intervals, "\n") # Add this as an exercise, send users here https://pystac.readthedocs.io/en/stable/api/collection.html
# Print out the assets and some of their attributes
for key, asset in collection.assets.items():
    print("Asset:", key)
    print("  href:", asset.get_absolute_href())
    print("  title:", asset.title)
    print("  description:", asset.description)

# print("Assets url:", collection.assets["daily-items-geoparquet"].get_absolute_href())

Collection ID :  noaa-cdr-sea-ice-concentration 

Collection Title: Sea Ice Concentration CDR 

Collection Description: The Sea Ice Concentration Climate Data Record (CDR) provides a consistent daily and monthly time series of sea ice concentrations for both the north and south Polar Regions on a 25 km x 25 km grid. These data can be used to estimate how much of the ocean surface is covered by ice, and monitor changes in sea ice concentration. The CDR combines concentration estimates using two algorithms developed at the NASA Goddard Space Flight Center (GSFC). Gridded brightness temperatures acquired from Defense Meteorological Satellite Program (DMSP) passive microwave radiometers, and, beginning 1 January 2025, the JAXA AMSR2 sensor, provide the necessary input to produce the dataset. 

STAC Version:  1.1.0 

Asset: daily-items-geoparquet
  href: https://noaadata.apps.nsidc.org/NOAA/G02202_V6/stac/sea-ice-concentration/sea-ice-concentration-daily-items.parquet
  title: Daily STAC It

## Load the STAC GeoParquet *Collection* Assets and Extract the *Items*

As you saw in the Cells 3 and 4 above, the *Collection* has two assets: one GeoParquet file for daily items and one for monthly. You also saw that the daily geoparquet asset is listed before the monthly one, so you will use that knowledge to extract the two URLs.
Each row in these assets is one STAC *Item* (one file, one hemisphere, one day or month). In the next cells, you will load these and see what *Items*  and attributes they contain.

<div class="alert alert-block alert-info">
<strong>Note:</strong> PySTAC does not natively read GeoParquet files out-of-the-box, so you will need <code>geopandas</code> to read them and then convert the resulting *Items* to a <code>pystac item object</code>.
</div>

In [5]:
# Cell 5
# Obtain the GeoParquet URLs from the Collection
urls = []
for key, asset in collection.assets.items():
    urls.append(asset.get_absolute_href())
daily_url = urls[0] # The daily geoparquet is first in the collection so it has an index of 0
monthly_url = urls[1] # The monthly geoparquet is second in the collection so it has an index of 1

# Set up the file system
fs = HTTPFileSystem()

# Read the daily and monthly geoparquet files and extract the items into two geopandas data frames
daily_items_gdf = gpd.read_parquet(daily_url, filesystem=fs)
monthly_items_gdf = gpd.read_parquet(monthly_url, filesystem=fs)

# Print out the number of items in each geoparquet file
print(f"Daily items:   {len(daily_items_gdf):,}")
print(f"Monthly items: {len(monthly_items_gdf):,}")

Daily items:   35,092
Monthly items: 1,152


In [24]:
# Cell 6
# Print the number of attributes retrieved from the daily geoparquet file and their names
print("Number of attributes: ", len(daily_items_gdf.columns))
print("Attribute names: ", daily_items_gdf.columns)

# View the first 5 rows of the dataframe to see the attributes and some of their values
daily_items_gdf.head()

Number of attributes:  17
Attribute names:  Index(['type', 'stac_version', 'stac_extensions', 'id', 'geometry', 'links',
       'assets', 'start_datetime', 'end_datetime', 'noaa_cdr:interval',
       'processing:level', 'proj:epsg', 'proj:wkt2', 'proj:shape',
       'proj:transform', 'noaa_cdr:hemisphere', 'datetime'],
      dtype='str')


,type,stac_version,stac_extensions,id,geometry,links,assets,start_datetime,end_datetime,noaa_cdr:interval,processing:level,proj:epsg,proj:wkt2,proj:shape,proj:transform,noaa_cdr:hemisphere,datetime
0,Feature,1.0.0,[https://stac-extensions.github.io/processing/...,sic_psn25_19781025_n07_v06r00,"POLYGON ((180 30.98056, 180 90, -180 90, -180 ...",[],{'netcdf': {'href': 'https://noaadata.apps.nsi...,1978-10-25 00:00:00+00:00,1978-10-25 23:59:59+00:00,daily,L3,None,"PROJCRS[""unknown"",BASEGEOGCRS[""unknown"",DATUM[...","[448, 304]","[25000.0, 0.0, -3850000.0, 0.0, -25000.0, 5850...",north,NaT
1,Feature,1.0.0,[https://stac-extensions.github.io/processing/...,sic_psn25_19781026_n07_v06r00,"POLYGON ((180 30.98056, 180 90, -180 90, -180 ...",[],{'netcdf': {'href': 'https://noaadata.apps.nsi...,1978-10-26 00:00:00+00:00,1978-10-26 23:59:59+00:00,daily,L3,None,"PROJCRS[""unknown"",BASEGEOGCRS[""unknown"",DATUM[...","[448, 304]","[25000.0, 0.0, -3850000.0, 0.0, -25000.0, 5850...",north,NaT
2,Feature,1.0.0,[https://stac-extensions.github.io/processing/...,sic_psn25_19781027_n07_v06r00,"POLYGON ((180 30.98056, 180 90, -180 90, -180 ...",[],{'netcdf': {'href': 'https://noaadata.apps.nsi...,1978-10-27 00:00:00+00:00,1978-10-27 23:59:59+00:00,daily,L3,None,"PROJCRS[""unknown"",BASEGEOGCRS[""unknown"",DATUM[...","[448, 304]","[25000.0, 0.0, -3850000.0, 0.0, -25000.0, 5850...",north,NaT
3,Feature,1.0.0,[https://stac-extensions.github.io/processing/...,sic_psn25_19781028_n07_v06r00,"POLYGON ((180 30.98056, 180 90, -180 90, -180 ...",[],{'netcdf': {'href': 'https://noaadata.apps.nsi...,1978-10-28 00:00:00+00:00,1978-10-28 23:59:59+00:00,daily,L3,None,"PROJCRS[""unknown"",BASEGEOGCRS[""unknown"",DATUM[...","[448, 304]","[25000.0, 0.0, -3850000.0, 0.0, -25000.0, 5850...",north,NaT
4,Feature,1.0.0,[https://stac-extensions.github.io/processing/...,sic_psn25_19781029_n07_v06r00,"POLYGON ((180 30.98056, 180 90, -180 90, -180 ...",[],{'netcdf': {'href': 'https://noaadata.apps.nsi...,1978-10-29 00:00:00+00:00,1978-10-29 23:59:59+00:00,daily,L3,None,"PROJCRS[""unknown"",BASEGEOGCRS[""unknown"",DATUM[...","[448, 304]","[25000.0, 0.0, -3850000.0, 0.0, -25000.0, 5850...",north,NaT


## Filter *Items*

Because there are over 35,000 *Items* in the daily geoparquet, filter by date and hemisphere so you are working with a smaller number of them.
This is fast even for the full archive because it operates on columnar data in memory. You will filter for January 2025 for the Northern Hemisphere using the attribute names found in Cell 6. Then, print the number of retrieved *Items*.

In [7]:
# Cell 7
# Filter for a specific date range and hemisphere
daily_items_gdf_filter = daily_items_gdf[(daily_items_gdf['start_datetime'] >= '2025-01-01') & 
                         (daily_items_gdf['start_datetime'] <= '2025-01-31') & (daily_items_gdf['noaa_cdr:hemisphere'] == 'north')]

# Confirm that 31 rows were retrieved for the 31 days of January
print(len(daily_items_gdf_filter))

31


## Convert dataframe of *Items* to pystac item object
Convert the extracted *Items* from the geopandas dataframe to a `pystac item object`, so you can use pystac operations to work with the data.

In [8]:
# Cell 8
# Using stac_geoparquet convert dataframe to pystac object
items_jan2025_nh = stac_geoparquet.to_item_collection(daily_items_gdf_filter)

<div class="alert alert-block alert-success">
<h3><strong>Explore</strong></h3>
As you have seen previously with the <em>Collection</em> metadata, you will see similar output when you view the metadata for a group of <em>Items</em> or a single <em>Item</em>. Again, the lines with a triangle next to them are clickable and reveal more metadata elements underneath. Let's explore the <em>Items</em>. There are multiple ways to do this:
<ul>
    <li>As with the <code>collection</code>, you can start by simply calling the instance of the <code>pystac item object</code> and see that there are 31 <em>Items</em> in the object (Cell 9), which matches the number found in Cell 7. You can interact with the object via the triangles.</li>
    <li>You can view all the <em>Item</em> ids (Cell 10).</li>
    <li>You can look at just the first <em>Item</em> using the index 0 (Cell 11). You can interact with the object via the triangles.</li>
    <li>You can extract and print out only the specific metadata that you are interested in for the first item. In Cell 12, you will print out the <em>Item</em> <code>id</code>, <code>start_datetime</code>, and <code>assets</code> attributes.</li>
</ul>
</div>

<div class="alert alert-block alert-info">
<strong>Note:</strong> Pay close attention to the format of the <code>item id</code> because you will need it later. You can see that the id contains the date of the data in the form <code>YYYYMMDD</code>. You will use that to search for specific ids.
</div>

In [9]:
# Cell 9
# Call instance of the pystac item object to see all attributes
items_jan2025_nh

In [10]:
# Cell 10
# View all the item_ids
items_jan2025_nh.items

[<Item id=sic_psn25_20250101_am2_v06r00>,
 <Item id=sic_psn25_20250102_am2_v06r00>,
 <Item id=sic_psn25_20250103_am2_v06r00>,
 <Item id=sic_psn25_20250104_am2_v06r00>,
 <Item id=sic_psn25_20250105_am2_v06r00>,
 <Item id=sic_psn25_20250106_am2_v06r00>,
 <Item id=sic_psn25_20250107_am2_v06r00>,
 <Item id=sic_psn25_20250108_am2_v06r00>,
 <Item id=sic_psn25_20250109_am2_v06r00>,
 <Item id=sic_psn25_20250110_am2_v06r00>,
 <Item id=sic_psn25_20250111_am2_v06r00>,
 <Item id=sic_psn25_20250112_am2_v06r00>,
 <Item id=sic_psn25_20250113_am2_v06r00>,
 <Item id=sic_psn25_20250114_am2_v06r00>,
 <Item id=sic_psn25_20250115_am2_v06r00>,
 <Item id=sic_psn25_20250116_am2_v06r00>,
 <Item id=sic_psn25_20250117_am2_v06r00>,
 <Item id=sic_psn25_20250118_am2_v06r00>,
 <Item id=sic_psn25_20250119_am2_v06r00>,
 <Item id=sic_psn25_20250120_am2_v06r00>,
 <Item id=sic_psn25_20250121_am2_v06r00>,
 <Item id=sic_psn25_20250122_am2_v06r00>,
 <Item id=sic_psn25_20250123_am2_v06r00>,
 <Item id=sic_psn25_20250124_am2_v

In [11]:
# Cell 11
# View the attributes of the firts item in the pystac object
items_jan2025_nh[0]

<Item id=sic_psn25_20250101_am2_v06r00>

In [12]:
# Cell 12
# Look at the first item
first_item = items_jan2025_nh[0]
print("Item ID:", first_item.id)
print("start date:", first_item.properties["start_datetime"])
print("Assets:", first_item.assets["netcdf"].href)

Item ID: sic_psn25_20250101_am2_v06r00
start date: 2025-01-01T00:00:00.000000Z
Assets: https://noaadata.apps.nsidc.org/NOAA/G02202_V6/north/daily/2025/sic_psn25_20250101_am2_v06r00.nc


#### Extract the 15 January 2025 *Item*
You can use the <code>Item ID</code> to extract and examine the metadata for a specific <em>Item</em> (Cells 13 and 14). In the example below, you will extract the *Item* for 15 January 2025 by its ID and explore it.

To do this, you will do the following steps:
* Extract all the `item ids` from the `items_jan2025_nh` object created in Cell 8.
* Find the `item id` for the 15 January 2025 *Item*. Since you know the form of the `item ids` from cell 10, you will search the `item_ids` for the date `"20250115"`
* Loop through the *Items* to find the one that matches the ID you are interested in

In [13]:
# Cell 13
# Extract all the Item IDs from the Item list
jan2025_item_ids = [i.id for i in items_jan2025_nh]

# Find the Item id for 15 January 2025
# This command returns a list with one element that contains the id you want
jan15_item_id = [i for i in jan2025_item_ids if "20250115" in i]

# Get Item using Item id
# Returns the matching Item, or None if not found
jan15_item = next((item for item in items_jan2025_nh if item.id == jan15_item_id[0]), None)

# Print if an item was found or not
if jan15_item:
    print(f"Found item: {jan15_item.id}")
else:
    print("Item not found.")

Found item: sic_psn25_20250115_am2_v06r00


#### Print *Item* Metadata
As seen in other `pystac` objects, the `jan15_item` contains metadata in the form of properties, attributes, and assets. You will explore some of these by printing them.

First, look at all metadata for the `jan15_item`. Again, you can interact with the triangles to see more metadata.

In [14]:
# Cell 14
# Show all properites, attributes, and assets by looking at the item instance
jan15_item

<Item id=sic_psn25_20250115_am2_v06r00>

#### Print Out the *Item* Properties
You can also look at just the *Item* properties with the following command.

In [15]:
# Cell 15
# Print out Item properties
jan15_item.properties

{'start_datetime': '2025-01-15T00:00:00.000000Z',
 'end_datetime': '2025-01-15T23:59:59.000000Z',
 'noaa_cdr:interval': 'daily',
 'processing:level': 'L3',
 'proj:epsg': None,
 'proj:wkt2': 'PROJCRS["unknown",BASEGEOGCRS["unknown",DATUM["unknown",ELLIPSOID["unknown",6378273,298.279411123064,LENGTHUNIT["metre",1,ID["EPSG",9001]]]],PRIMEM["Greenwich",0,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8901]]],CONVERSION["unknown",METHOD["Polar Stereographic (variant B)",ID["EPSG",9829]],PARAMETER["Latitude of standard parallel",70,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8832]],PARAMETER["Longitude of origin",-45,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8833]],PARAMETER["False easting",0,LENGTHUNIT["metre",1],ID["EPSG",8806]],PARAMETER["False northing",0,LENGTHUNIT["metre",1],ID["EPSG",8807]]],CS[Cartesian,2],AXIS["(E)",south,MERIDIAN[90,ANGLEUNIT["degree",0.0174532925199433,ID["EPSG",9122]]],ORDER[1],LENGTHUNIT["metre",1,ID["EPSG",9001]]],AXIS["(N)",south,MERIDIAN[180,ANG

#### Print Out Other *Item* Properties and Attributes
You can print specific properties and attributes with the next commands.

In [16]:
# Cell 16
# Print out specific Item metadata
print('Geometry: ', jan15_item.geometry)
print('Geospatial Bounding Box: ', jan15_item.bbox)
print('Start date: ', jan15_item.properties['start_datetime'])
print('End date: ', jan15_item.properties['end_datetime'])

Geometry:  {'type': 'Polygon', 'coordinates': (((180.0, 30.980564), (180.0, 90.0), (-180.0, 90.0), (-180.0, 30.980564), (180.0, 30.980564)),)}
Geospatial Bounding Box:  None
Start date:  2025-01-15T00:00:00.000000Z
End date:  2025-01-15T23:59:59.000000Z


<mark>Have users read the monthly data instead??</mark>
<div class="alert alert-block alert-success">
<h3><strong>Exercise 1</strong></h3>
You will use the monthly *Collection Item* (monthly_items_gdf) loaded in Cell 5 to complete this exercise. 
Using the examples from Tutorial Part 1, uncomment the code below, and change the code as necessary to extract and explore the metadata for the Southern Hemisphere <em>Item</em> for July 2012.</br></br>

<div class="alert alert-block alert-info">
<strong>Note:</strong>  Variable names have been updated for you but spots where you need to input dates, hemisphere, or attribute names have not.
</div>

<div class="alert alert-block alert-info">
<strong>Hint:</strong> Think about what you need to search for to find the correct <code>Item id</code> for the monthly July 2012 Southern Hemisphere <em>Item</em>.
</div>


In [ ]:
# Cell 17
# Uncomment out the two lines below and update the date and hemisphere as necessary to filter for the month of July 2012 for southern hemisphere
#daily_items_gdf_filter_ex1 = daily_items_gdf[(daily_items_gdf['start_datetime'] >= '2025-01-01') & 
#                            (daily_items_gdf['start_datetime'] <= '2025-01-31') & (daily_items_gdf['noaa_cdr:hemisphere'] == 'hemi')]

# Using stac_geoparquet convert dataframe to pystac object
#items_jul2012_sh = stac_geoparquet.to_item_collection(daily_items_gdf_filter_ex1) # Uncomment out this command and update if necessary

# Show the group of items
#items_jul2012_sh # Uncomment out this command and update if necessary

In [ ]:
# Cell 18
# Extract all the Item IDs from the Item list
#jul2012_item_ids = [i.id for i in items_jul2012_sh] # Uncomment out this command and update if necessary

# Find the Item id for 25 July 2012 by updating YYYYMMDD in the quotes
# This command returns a list with one element that contains the id you want
#jul25_item_id = [i for i in jul2012_item_ids if "YYYYMMDD" in i] # Uncomment out this command and update as necessary

# Get Item using Item id
# Returns the matching Item, or None if not found
#jul25_item = next((item for item in items_jul2012_sh if item.id == jul25_item_id[0]), None) # Uncomment out this command and update if necessary

# Print if item was found or not
# Uncomment out the flowing 4 lines and update if necessary
#if jul25_item:
#    print(f"Found item: {jul25_item.id}")
#else:
#    print("Item not found.")

# Show all properites, attributes, and assets by looking at the item instance
# After this line runs, click the triangle next to the properties attribute and confirm that the start_datetime is 2012-07-25
#jul25_item # Uncomment out this command and update if necessary

#### Check your work
After you run Cell 19, confirm that the start date is 2012-07-25

In [ ]:
# Cell 19
# Uncomment out this command and update as necessary
#print('Geometry: ', jul25_item.geometry) # Uncomment out this command and update as necessary
#print('Geospatial Bounding Box: ', jul25_item.bbox) # Uncomment out this command and update as necessary
#print('Start date: ', jul25_item.properties['start_datetime']) # Uncomment out this command and update as necessary
#print('End date: ', jul25_item.properties['end_datetime']) # Uncomment out this command and update as necessary

In [ ]:
# Delete when done testing
# Uncomment out the two lines below and update the date and hemisphere as necessary to filter for a specific date range and hemisphere
monthly_items_gdf_filter_exc1 = monthly_items_gdf[(monthly_items_gdf['start_datetime'] >= '2012-07-01') & 
                            (daily_items_gdf['start_datetime'] <= '2012-07-31') & (daily_items_gdf['noaa_cdr:hemisphere'] == 'south')]

# Using stac_geoparquet convert dataframe to pystac object
items_jul2012_sh = stac_geoparquet.to_item_collection(daily_items_gdf_filter_ex1) # Uncomment out this command and update as necessary

# Show the group of items
items_jul2012_sh # Uncomment out this command and update as necessary

In [ ]:
# Delete when done texting
# Extract all the Item IDs from the Item list
jul2012_item_ids = [i.id for i in items_jul2012_sh] # Uncomment out this command and update if necessary

# Find the Item id for 25 July 2012 by updating the date in the quotes
# This command returns a list with one element that contains the id you want
jul25_item_id = [i for i in jul2012_item_ids if "20120725" in i] # Uncomment out this command and update as necessary

# Get Item using Item id
# Returns the matching Item, or None if not found
jul25_item = next((item for item in items_jul2012_sh if item.id == jul25_item_id[0]), None) # Uncomment out this command and update if necessary

# Print if item was found or not
# Uncomment out the flowing 4 lines and update if necessary
if jul25_item:
    print(f"Found item: {jul25_item.id}")
else:
    print("Item not found.")

# Show all properites, attributes, and assets by looking at the item instance
jul25_item # Uncomment out this command and update if necessary

In [ ]:
# Delete this when done testing
# Uncomment out the following lines and update as necessary
print('Geometry: ', jul25_item.geometry) # Uncomment out this command and update as necessary
print('Geospatial Bounding Box: ', jul25_item.bbox) # Uncomment out this command and update as necessary
print('Start date: ', jul25_item.properties['start_datetime']) # Uncomment out this command and update as necessary
print('End date: ', jul25_item.properties['end_datetime']) # Uncomment out this command and update as necessary

<div class="alert alert-block alert-success">
<h3><strong>Bonus Question</strong></h3>
What command would you use to view the netCDF assets? Add this command to the blank line in the code Cell 20 below. You'll be using this in Part 2 of the tutorial below, so if you don't know the answer, you can find it there.
</div>

<div class="alert alert-block alert-info">
<strong>Hint:</strong> If you are unsure about what attributes are available, print out the <em>Item</em> instance again. The command will look something like <code>item_name.attribute['element']</code>.
</div>

In [ ]:
# Cell 20
# Add a command to view the netCDF assets here


************************
## Tutorial Part 2

Read NetCDF File from the STAC
************************

### Read Data from NetCDF File Using STAC
Now that you have explored the SIC CDR STAC *Collection* and its *Items* in part 1 of this tutorial, you can use this information to read the netCDF data file associated with an *Item*. You will read the data from the 15 January 2025 Northern Hemisphere *Item* (`jan15_item`) extracted in Cell 13.

#### Print out the NetCDF Assets
In Part 1 of this tutorial, you may have seen that the STAC *Item* contains a metadata attribute called `assets` that give you some information about the netCDF file, including the location of the file via its URL in the `href` attribute.

<div class="alert alert-block alert-info">
<strong>Note:</strong> The command in Cell 21 is the answer to the Bonus Question from Exercise 1.
</div>

In [ ]:
# Cell 21
# Access the NetCDF assets from the STAC *Item* for 15 January 2025 and see the components
jan15_item.assets['netcdf']

#### Extract the URL from the Assets and Read the NetCDF File

You will use fsspec and xarray to read the netCDF file. To get the URL of the netCDF file, you will extract the `href` attribute from the netCDF assets. 

In [ ]:
# Cell 22
# Get the URL from the STAC Item assets and print it out
netcdf_url_jan15 = jan15_item.assets['netcdf'].href
print("NetCDF URL: ", netcdf_url_jan15)

# Read the netCDF file at that URL using fsspec and xarray
fs_jan15 = fsspec.filesystem('https')
ds_jan15 = xr.open_dataset(fs_jan15.open(netcdf_url_jan15))

#### Explore the NetCDF Dataset and its Variables
<mark>Double check if this needs to be updated</mark>

You can view all the variables within the netCDF file by calling the dataset instance. Xarray provides a nice interactive way to explore the file contents. You can see the coordinates for the data (`time`, `xgrid`, `ygrid`) and then the variables themselves (`cdr_sesaice_conc`, `projection`, `qa_of_cdr_seaice_conc`, `stdev_of_cdr_seaice_conc`, etc.).

<div class="alert alert-block alert-success">
<h3><strong>Explore</strong></h3>
After running the command in Cell 23 below, you will see information about the netCDF data formatted with Xarray. The first line, `Dimensions`, provides the dimensions of the dataset. In this case, `tdim` is the time dimension with 365 values (one for each days of the year), the y axis with 332 elements, and the x axis with 316 elements.</br></br>
This output also provides information about the netCDF variables. Each netCDF variable has two icons to its right: a file icon and a database icon (looks like 3 coins stacked on top of each other).
    <ul>
    <li>File icon: Clicking this lists the variable attributes available for that variable (for example: long_name, standard_name, flag_values, units, valid range, etc.). These variable attributes can help you determine what a variable is and how to use it. Note that not all variables have the same attributes. These are dependent on what the data creator supplied with the netCDF data files.</li>
    <li>Database icon: Clicking this shows you the number of elements in the variable and its data type.</li>
    </ul>
Click on the icons to explore the variable attributes of the netCDF variables. <strong>Note:</strong> To close the content, you have to click the icon again.
</br></br>
In addition to variable attributes, the netCDF files also contain global attributes. These can be viewed by clicking on the word `Attributes` in the xarray output. You will see things like title, summary, and spatial and temporal resolution, to name a few. 
</div>

In [ ]:
# Cell 23
# Print out the dataset instance to see its contents
ds_jan15

#### Access Specific NetCDF Global Attributes
You can print out specific global attributes by calling them by name from the dataset instance. As noted above, you can find the name of these attributes by clicking `Attributes` in the code output above. In Cell 24 below, you will print out the `spatial_resolution`, `platform`, and `sensor` from the global attributes.

In [ ]:
# Cell 24
# Print some global attributes
print("Spatial Resolution: ", ds_jan15.spatial_resolution)
print("Platform: ", ds_jan15.platform)
print("Sensor: ", ds_jan15.sensor)

#### Other Ways to Explore the NetCDF Variables
To print out a simple list of the netCDF variables, you can use the `data_vars` property of the xarray dataset. One of the variables is called `cdr_seaice_conc`. This contains the sea ice data you are interested in.

In [ ]:
# Cell 25
# Print out simple list of netCDF variables
ds_jan15.data_vars

#### Print Out All Metadata for the Variables
You can view all the information at once about the netCDF variables using the `keys` method on the `variables` property.
<div class="alert alert-block alert-warning">
    <h3>Warning</h3>
    This prints a rather messy list of all the variables and their attributes.
</div>

In [ ]:
# Cell 26
# Take a look at the variable keys for the netCDF dataset
ds_jan15.variables.keys()

#### Extract the Sea Ice Concentration Data

You saw from the command above that the sea ice concentration variable is called `cdr_seaice_conc`. 
You will use this to extract the data from the file into a variable.

In [ ]:
# Cell 27
# Pull out sea ice concentration variable as an xarray object
cdr_seaice_conc_jan15 = ds_jan15.variables['cdr_seaice_conc']

#### Explore the Attributes of the cdr_seaice_conc Variable
Just as you were able to view the attribute information for the variables in the xarray dataset object `ds_2023`, you can also view the attributes specifically for the variable once you've extracted it. As with the xarray dataset, viewing the properties of the xarray variable can be done in several ways. You will look at two of them: calling the xarray variable instance and using the `attrs` property of the xarray varaible.

<div class="alert alert-block alert-info">
<strong>Note:</strong> You'll need some of this information in Notebook 3 when you plot the data.
</div>

**Using the Xarray Variable Instance**

You can simply call the name of the xarray variable to view its properties.

<div class="alert alert-block alert-success">
<h3><strong>Explore</strong></h3>
Click on the word <code>Attributes</code> and explore the attributes associated with the variable.
</div>

In [ ]:
# Cell 28
# Print out the cdr_seaice_conc_2023 Xarray variable instance
cdr_seaice_conc_jan15

**Using the attrs Property**

You can simply print the attributes as text using the `attrs` property.

In [ ]:
# Cell 29
# Print attributes using the atrs property
cdr_seaice_conc_jan15.attrs

<div class="alert alert-block alert-success">
<h3><strong>Exercise 2</strong></h3>
Using the example from Tutorial Part 2, uncomment the template below, and change the code as necessary to read the 25 July 2012 *Item* for the Southern Hemisphere and extract the temporal interpolation flag variable and explore its contents.</br></br>
<strong>Be sure to update variable names as appropriate.</strong>
</div>

<div class="alert alert-block alert-warning">
    <h3>Warning</h3>
    You need to complete Exercise 1 for the code in Exercise 2 to work.
</div>

<mark>Check this below to make sure it's correct. I'm not sure data_vars is a method</mark>
<div class="alert alert-block alert-info">
<strong>Hint:</strong> Use the <em>Item</em> variable you created in Exercise 1 to accomplish this task. Use the method object <code>data_vars</code> to find the name of the flag.
</div>

#### Read the 15 July 2012 Southern Hemisphere NetCDF File

In [ ]:
# Cell 30
# Get the URL from the STAC Item assets and print it out
netcdf_url_jan15 = jan15_item.assets['netcdf'].href # Uncomment out this command and update as necessary
print("NetCDF URL: ", netcdf_url_jan15) # Uncomment out this command and update as necessary

# Read the netCDF file at that URL using fsspec and xarray
fs_jan15 = fsspec.filesystem('https') # Uncomment out this command and update as necessary
ds_jan15 = xr.open_dataset(fs_jan15.open(netcdf_url_jan15)) # Uncomment out this command and update as necessary

In [ ]:
# Cell 31
# Print out simple list of netCDF variables and find the one for the temporal interpolation flag
ds_jan15.data_vars # Uncomment out this command and update as necessary

#### Extract Temporal Interpolation Flag

<mark>Which cell exactly?</mark>

<div class="alert alert-block alert-info">
<strong>Hint:</strong> From the listing of the variables in the previous code cell, you see that the netCDF variable for the temporal interpolatoin flag is called <code>cdr_seaice_conc_interp_temporal_flag</code>.
</div>

In [ ]:
tflag = ds_jan15.variables['cdr_seaice_conc_interp_temporal_flag']
tflag[0,0]

In [ ]:
# Cell 32
# Pull out temporal interploation flag into a variable as an xarray object
# Note: The variable name on the left (temporal_flag) has been set for you but you must update all the information on the right side of the equals sign
# with the correct xarray dataset name that you created in Cell 30 and add the correct variable name where it says netcdf_variable See Hint, above.
temporal_flag = ds_jan15.variables['netcdf_variable'] # Uncomment out this command and update as necessary

# Print out the attribute details of the temporal interpolation flag
temporal_flag # Uncomment out this command

### Next Steps

After successful completion of Notebook 2, you can go on to Notebook 3: Mapping Sea Ice Concentration.

### Data Statement

Meier, W. N., Fetterer, F., Windnagel, A. K., & Stewart, J. S. (2021). NOAA/NSIDC Climate Data Record of Passive Microwave Sea Ice Concentration. (G02202, Version 4). [Data Set]. Boulder, Colorado USA. National Snow and Ice Data Center. https://doi.org/10.7265/efmz-2t65.

### Metadata

* Language/Package(s)
  * Python
  * xarray
  * pystac
  * geopandas
* Geophysical Keywords
  * Sea Ice Concentration

### Software License

Copyright 2025 NOAA NCAI

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the “Software”), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED “AS IS”, WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.

### Content License

[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/ "Createive Commons BY 4.0 License")

### Disclaimer

This Jupyter notebook is a scientific product and is not official communication of the National Oceanic and Atmospheric Administration, the United States Department of Commerce, the National Snow and Ice Data Center, or the University of Colorado at Boulder. All NOAA Jupyter notebooks are provided on an 'as is' basis, and the user assumes responsibility for its use. Any claims against the Department of Commerce or Department of Commerce bureaus stemming from the use of this Jupyter notebook will be governed by all applicable Federal law. Any reference to specific commercial products, processes, or services by service mark, trademark, manufacturer, or otherwise does not constitute or imply their endorsement, recommendation or favoring by the Department of Commerce. The Department of Commerce seal and logo, or the seal and logo of a DOC bureau, shall not be used in any manner to imply endorsement of any commercial product or activity by DOC or the United States Government.